# Implementation of C-Mixup
This is the individual part for Nicolai Andersen

In [14]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Optional

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


Columns for our data

In [15]:
numeric_cols = ['quarter_label', 'deflated_gdp_usd', 'us_cpi', 'straight_distance_to_capital_km']
categorical_cols = ['geolocation_name', 'country', 'landlocked', 'region_economic_classification',
                    'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway',
                    'seismic_hazard_zone', 'flood_risk_class', 'tropical_cyclone_wind_risk',
                    'tornadoes_wind_risk', 'koppen_climate_zone']
image_col = 'processed_imgs'
target_col = 'construction_cost_per_m2_usd'

### Interpolation

In [16]:
def interpolate_images(a_path: str, b_path: str, lam: float):
    img_path = "../Processed data/processed_composite/"
    c_path = "C-Mixup/" + a_path[:-3] + '_' + b_path[:-3] + '.pt'

    a = torch.load(img_path + a_path, weights_only=True)
    a_sentinel = a['sentinel']
    a_viirs = a['viirs']

    b = torch.load(img_path + b_path, weights_only=True)
    b_sentinel = b['sentinel']
    b_viirs = b['viirs']
    
    sentinel_img = torch.lerp(b_sentinel, a_sentinel, lam)
    viirs_img = torch.lerp(b_viirs, a_viirs, lam)

    tensor = {'sentinel': sentinel_img, 'viirs': viirs_img}

    return c_path, tensor


def interpolate_datapoints(a: pd.Series, b: pd.Series, numeric_cols: list, categorical_cols: list, img_col: str, target_col, lam: float) -> pd.Series:
    """
    Interpolate two datapoints and all their values according to their type.
 
    numeric / tensor  →  lam*a + (1-lam)*b
    categorical       →  a if lam >= 0.5 else b   (dominant-sample rule)
    """
    c = pd.Series()
    keys = a.keys()
    numeric_cols = [col for col in numeric_cols if col in keys]
    categorical_cols = [col for col in categorical_cols if col in keys]
    
    for col in numeric_cols:
        c[col] = lam * a[col] + (1 - lam) * b[col]
    for col in categorical_cols:
        c[col] = a[col] if lam >= 0.5 else b[col]
    c[img_col], tensor = interpolate_images(a[img_col], b[img_col], lam)
    c[target_col] = lam * a[target_col] + (1 - lam) * b[target_col]
    
    return c, tensor

### Sampling

In [17]:
def get_sampling_probs(df: pd.DataFrame, t_col: str, sigma: float) -> np.ndarray:
    """
    Build an NxN matrix of sampling probabilities

    P[i, j] = softmax_j( -||y_i - y_j||**2 / (2 * sigma ** 2))
    """
    y_array = df[t_col].to_numpy().reshape(-1, 1)
    n = len(y_array)
    sq_norms = np.sum(y_array ** 2, axis=1, keepdims=True) # (n, 1)
    dists_sq = sq_norms + sq_norms.T - 2.0 * (y_array @ y_array.T)
    dists_sq = np.clip(dists_sq, 0.0, None)

    log_probs = -dists_sq / (2.0 * sigma ** 2)
    np.fill_diagonal(log_probs, -np.inf) #Exclude self

    log_probs -= log_probs.max(axis=1, keepdims=True) # Subtract max per row
    probs = np.exp(log_probs)
    row_sums = probs.sum(axis=1, keepdims=True)
    probs /= row_sums

    return probs

### Public API
for other files to access this method

In [ ]:
class CMixup:
    """
    C-Mixup data augmentation for regression.

    Parameters
    ----------
    label_col : str
        Name of the target / label column in the DataFrame.
    sigma : float
        Gaussian kernel bandwidth σ.  Larger σ → closer to vanilla mixup
        (uniform sampling).  Smaller σ → only very similar labels are mixed.
        The paper recommends choosing via cross-validation; a good starting
        point is the standard deviation of the label values.
    alpha : float
        Shape parameter for the Beta(α, α) distribution from which the
        interpolation ratio λ is sampled.  α = 0.2 is a common default.
    ratio_of_samples : float
        Ratio of how many samples are taken from training data when creating
        new synthetic samples.
    include_original : bool
        If True (default) the returned DataFrame contains the original rows
        followed by the synthetic rows.
    seed : int or None
        Random seed for reproducibility.
    tensor_cols : list[str] or None
        Explicit list of columns that contain torch.Tensor / np.ndarray
        objects.  If None the class auto-detects them from the first row.
    categorical_cols : list[str] or None
        Explicit list of categorical columns.  If None the class
        auto-detects object / Categorical dtype columns.
    """

    def __init__(
        self,
        label_col: str,
        sigma: float = 1.0,
        alpha: float = 0.2,
        ratio_of_samples: int = 1,
        include_original: bool = False,
        seed: Optional[int] = None,
        tensor_col: str = None,
        categorical_cols: List[str] = None,
        numeric_cols: List[str] = None,
    ):
        if sigma <= 0: raise ValueError("sigma must be positive")
        if alpha <= 0: raise ValueError("alpha must be positive")
        if 1.0 < ratio_of_samples < 0.0: raise ValueError("ratio_of_samples must be <= 1.0 and >= 0.0")

        self.label_col = label_col
        self.sigma = sigma
        self.alpha = alpha
        self.ratio_of_samples = ratio_of_samples
        self.include_original = include_original
        self.seed = seed
        self._tensor_col = tensor_col
        self._cat_cols = categorical_cols
        self._num_cols = numeric_cols
        
        # Set after fit()
        self._sampling_probs: Optional[np.ndarray] = None   # (N, N)
        self._df_fit: Optional[pd.DataFrame] = None

    def get_sampling_probs(self, df: pd.DataFrame) -> np.ndarray:
        """
        Build an NxN matrix of sampling probabilities

        P[i, j] = softmax_j( -||y_i - y_j||**2 / (2 * sigma ** 2))
        """
        if self.label_col not in df.columns: raise ValueError(f"label_col '{self.label_col}' not found in DataFrame")

        y_array = df[self.label_col].to_numpy().reshape(-1, 1)
        n = len(y_array)
        sq_norms = np.sum(y_array ** 2, axis=1, keepdims=True) # (n, 1)
        dists_sq = sq_norms + sq_norms.T - 2.0 * (y_array @ y_array.T)
        dists_sq = np.clip(dists_sq, 0.0, None)

        log_probs = -dists_sq / (2.0 * self.sigma ** 2)
        np.fill_diagonal(log_probs, -np.inf) #Exclude self

        log_probs -= log_probs.max(axis=1, keepdims=True) # Subtract max per row
        probs = np.exp(log_probs)
        row_sums = probs.sum(axis=1, keepdims=True)
        probs /= row_sums

        return probs
    
    def fit(self, df: pd.DataFrame) -> "CMixup":
        """
        Compute and cache the label-similarity kernel on the training data.

        Parameters
        ----------
        df : pd.DataFrame
            Training data including the label column.

        Returns
        -------
        self
        """
        self._sampling_probs = self.get_sampling_probs(df)
        self._df_fit = df.reset_index(drop=True)
        return self
    
    def transform(
            self,
            df: pd.DataFrame,
            ratio_of_samples: Optional[float] = None,
    ) -> pd.DataFrame:
        """
        Generate synthetic samples by C-Mixup interpolation.

        If called after fit() the *fitted* kernel is used (transductive
        mode – mixing is relative to the training set).  If df is the
        same object passed to fit() this is the standard in-distribution
        augmentation.

        Parameters
        ----------
        df : pd.DataFrame
            Data to augment (typically the training set).
        ratio_of_samples : float or None
            Override self.ratio_of_samples for this call.

        Returns
        -------
        pd.DataFrame
            Original rows (if include_original=True) followed by synthetic
            rows.  Index is reset.
        """
        if self._sampling_probs is None:
            raise RuntimeError("Call fit() before transform()")
        ratio = ratio_of_samples if ratio_of_samples is not None else self.ratio_of_samples
        size = int(len(df) * ratio)
        rng = np.random.default_rng(self.seed)
        df = df.reset_index(drop=True)

        #Sampling batch from training data
        batch_indices = rng.choice(len(df), size=size, replace=False)
        lambdas = rng.beta(self.alpha, self.alpha, size=size)

        synthetic_rows = []
        tensor_dict = {}

        for idx, i in enumerate(batch_indices):
            lam = float(lambdas[idx])
            
            # Drawing mixing partner j
            probs_i = self._sampling_probs[i]

            j = int(rng.choice(len(probs_i), p=probs_i))
            
            row_i = df.iloc[i]
            row_j = self._df_fit.iloc[j]


            # Interpolate features
            new_row, new_tensor = interpolate_datapoints(
                row_i,
                row_j,
                self._num_cols,
                self._cat_cols,
                self._tensor_col,
                self.label_col,
                lam
            )
            synthetic_rows.append(new_row)
            tensor_dict[new_row[self._tensor_col]] = new_tensor


        df_synthetic = pd.DataFrame(synthetic_rows, columns=df.columns)

        if self.include_original:
            return pd.concat([df, df_synthetic], ignore_index=True)
        return df_synthetic.reset_index(drop=True), tensor_dict
    
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit on df and immediately return the augmented DataFrame."""
        return self.fit(df).transform(df)

    def __repr__(self) -> str:
        fitted = self._sampling_probs is not None
        return (
            f"CMixup(label_col={self.label_col!r}, sigma={self.sigma}, "
            f"alpha={self.alpha}, ratio_of_samples={self.ratio_of_samples}, "
            f"fitted={fitted})"
        )
    
csv = pd.read_csv("../Processed data\processed_japan.csv")
print(csv.shape)

c = CMixup(
    label_col=target_col,
    sigma=1.0,
    alpha=0.2,
    ratio_of_samples=0.1,
    include_original=False,
    seed = None,
    tensor_col=image_col,
    categorical_cols=categorical_cols,
    numeric_cols=numeric_cols
)
c = c.fit(csv)
synth_data, synth_tensors = c.transform(csv)

print(synth_data.shape)
print(synth_tensors.keys())

c.__repr__()

(567, 17)
(56, 17)
56
dict_keys(['C-Mixup/27000_osaka_2020-Q3_28000_hyogo_2020-Q3.pt', 'C-Mixup/42000_nagasaki_2021-Q2_31000_tottori_2020-Q1.pt', 'C-Mixup/11000_saitama_2023-Q4_09000_tochigi_2019-Q4.pt', 'C-Mixup/13000_tokyo_2023-Q2_26000_kyoto_2020-Q4.pt', 'C-Mixup/15000_niigata_2021-Q3_42000_nagasaki_2023-Q1.pt', 'C-Mixup/47000_okinawa_2024-Q1_03000_iwate_2021-Q1.pt', 'C-Mixup/40000_fukuoka_2021-Q1_33000_okayama_2019-Q4.pt', 'C-Mixup/35000_yamaguchi_2022-Q2_29000_nara_2024-Q3.pt', 'C-Mixup/15000_niigata_2020-Q2_38000_ehime_2020-Q1.pt', 'C-Mixup/14000_kanagawa_2020-Q3_13000_tokyo_2022-Q4.pt', 'C-Mixup/31000_tottori_2023-Q1_24000_mie_2024-Q1.pt', 'C-Mixup/23000_aichi_2023-Q1_03000_iwate_2021-Q3.pt', 'C-Mixup/21000_gifu_2020-Q3_26000_kyoto_2022-Q2.pt', 'C-Mixup/37000_kagawa_2021-Q2_11000_saitama_2021-Q4.pt', 'C-Mixup/39000_kochi_2024-Q1_21000_gifu_2019-Q1.pt', 'C-Mixup/08000_ibaraki_2019-Q3_02000_aomori_2024-Q2.pt', 'C-Mixup/06000_yamagata_2024-Q1_34000_hiroshima_2023-Q3.pt', 'C-Mixup/4

"CMixup(label_col='construction_cost_per_m2_usd', sigma=1.0, alpha=0.2, ratio_of_samples=0.1, fitted=True)"